# Appendix F — From Macro State to Guarded-PIT BL Tilt

**Presentation-only boundary.** The guarded PIT cell
`factor_pit_ext2026` is the primary configuration. This notebook reads the same
separately completed `canonical_guard_ablation_reports.v1` bundle as Appendix E,
validates its manifest, exact completion-marker SHA, table inventory, schemas,
rows, hashes, and completed `factor_guard_ablation_run.v1` child lineage through
the pinned Factor and market manifests. It then renders producer-persisted macro
states and allocation transformations. It performs no provider/network calls,
finance-metric calculation, portfolio reconstruction, or canonical write.

`target_delta` is used for the allocation heatmap because it is the clearest
persisted final effect after BL conversion and the HRP/BL blend; intermediate raw
and applied tilts remain visible in the worked-date table.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import struct
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.colors import LinearSegmentedColormap

EXPECTED_BUNDLE_SCHEMA = "canonical_guard_ablation_reports.v1"
EXPECTED_SOURCE_RUN_SCHEMA = "factor_guard_ablation_run.v1"
EXPECTED_PRODUCER = "scripts/build_tear_sheet.py"
EXPECTED_TABLE_SCHEMAS = {
    "tear_sheet_factor_guard_ablation_ext2026.parquet": "tear_sheet.factor_guard_ablation.v1",
    "factor_guard_ablation_equity_ext2026.parquet": "factor_guard_ablation.equity.v1",
    "factor_guard_ablation_panel_ext2026.parquet": "factor_guard_ablation.panel.v1",
}
CONFIG_IDS = (
    "factor_pit_ext2026",
    "factor_pit_unguarded_diagnostic_ext2026",
    "factor_nonpit_diagnostic_ext2026",
    "factor_nonpit_unguarded_diagnostic_ext2026",
)
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
SECONDARY = "#52514e"
GRID = "#e1e0d9"


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Repository root not found; run this notebook inside the repository.")


def resolve_repo_path(repo_root: Path, raw: str | Path, label: str) -> Path:
    candidate = Path(raw).expanduser()
    if not candidate.is_absolute():
        candidate = repo_root / candidate
    resolved = candidate.resolve()
    try:
        resolved.relative_to(repo_root)
    except ValueError as exc:
        raise ValueError(f"{label} must remain inside repository root {repo_root}: {resolved}") from exc
    return resolved


def repo_relative_path(repo_root: Path, path: Path, label: str) -> str:
    try:
        return path.resolve().relative_to(repo_root.resolve()).as_posix()
    except ValueError as exc:
        raise ValueError(f"{label} must remain inside repository root {repo_root}: {path}") from exc


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def completed_manifest_sha256(marker_path: Path) -> str:
    lines = marker_path.read_text(encoding="utf-8").splitlines()
    if len(lines) != 1 or not lines[0].startswith("manifest_sha256="):
        raise ValueError(f"{marker_path} must equal one manifest_sha256=<64 hex> line")
    value = lines[0].removeprefix("manifest_sha256=")
    if lines[0] != f"manifest_sha256={value}" or len(value) != 64 or any(ch not in "0123456789abcdef" for ch in value):
        raise ValueError(f"Malformed exact completed-marker manifest SHA in {marker_path}")
    return value


def require_column(frame: pd.DataFrame, candidates: tuple[str, ...], context: str) -> str:
    matches = [name for name in candidates if name in frame.columns]
    if not matches:
        raise ValueError(f"{context} requires one of columns {candidates}; observed={list(frame.columns)}")
    return matches[0]


def table_entry(manifest: dict, filename: str) -> tuple[str, dict]:
    inventory = manifest.get("tables")
    if not isinstance(inventory, dict):
        raise ValueError("Bundle manifest requires a tables inventory mapping")
    matches = [(name, meta) for name, meta in inventory.items() if isinstance(meta, dict) and Path(str(meta.get("file", ""))).name == filename]
    if len(matches) != 1:
        raise ValueError(f"Manifest must inventory {filename} exactly once; matches={len(matches)}")
    return matches[0]


def validate_frame_inventory(repo_root: Path, frame: pd.DataFrame, path: Path, meta: dict, context: str) -> dict:
    observed_sha = sha256_file(path)
    if observed_sha != meta.get("sha256"):
        raise ValueError(f"{context} hash mismatch: manifest={meta.get('sha256')} observed={observed_sha}")
    if int(meta.get("rows", -1)) != len(frame):
        raise ValueError(f"{context} row-count mismatch: manifest={meta.get('rows')} observed={len(frame)}")
    expected_schema = EXPECTED_TABLE_SCHEMAS[path.name]
    schema = meta.get("schema", meta.get("schema_id"))
    if schema != expected_schema:
        raise ValueError(f"{context} schema mismatch: expected {expected_schema}, observed {schema}")
    return {"file": repo_relative_path(repo_root, path, context), "sha256": observed_sha, "rows": len(frame), "schema": expected_schema}


def validate_source_child_run_lineage(repo_root: Path, source_run_dir: Path, bundle_manifest: dict) -> dict:
    inputs = bundle_manifest.get("input_manifests")
    if not isinstance(inputs, dict) or set(inputs) != {"factor_guard_ablation_run"}:
        raise ValueError("Canonical bundle must pin exactly factor_guard_ablation_run")
    pinned = inputs["factor_guard_ablation_run"]
    manifest_path, marker_path = source_run_dir / "manifest.json", source_run_dir / "COMPLETED"
    if not manifest_path.is_file() or not marker_path.is_file():
        raise FileNotFoundError(f"Completed source guard-ablation run not found at {source_run_dir}")
    manifest_sha = sha256_file(manifest_path)
    if completed_manifest_sha256(marker_path) != manifest_sha:
        raise ValueError("Source child-run COMPLETED marker does not exactly bind manifest SHA")
    source_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if source_manifest.get("schema") != EXPECTED_SOURCE_RUN_SCHEMA or source_manifest.get("completed") is not True:
        raise ValueError(f"Source child run must be completed {EXPECTED_SOURCE_RUN_SCHEMA}")
    if (source_manifest.get("run_id"), manifest_sha) != (pinned.get("run_id"), pinned.get("manifest_sha256")):
        raise ValueError("Canonical bundle lineage diverges from source child-run identity/hash")
    parents = source_manifest.get("input_manifests")
    if not isinstance(parents, dict) or not {"factor_run", "market_snapshot"}.issubset(parents):
        raise ValueError("Source child run must pin parent Factor and market manifests")
    for role, identity_field in (("factor_run", "run_id"), ("market_snapshot", "snapshot_id")):
        declaration = parents[role]
        if not isinstance(declaration, dict) or not isinstance(declaration.get(identity_field), str) or len(str(declaration.get("manifest_sha256", ""))) != 64:
            raise ValueError(f"Source child-run parent lineage is malformed for {role}")
    source_files = source_manifest.get("files")
    source_artifacts = bundle_manifest.get("source_artifacts")
    if not isinstance(source_files, dict) or not isinstance(source_artifacts, dict) or not {"metric_records", "panel"}.issubset(source_artifacts):
        raise ValueError("Canonical bundle must hash-bind child metric-record and panel artifacts")
    for role, declaration in source_artifacts.items():
        matches = [entry for entry in source_files.values() if isinstance(entry, dict) and entry.get("file") == declaration.get("file")]
        source_path = (source_run_dir / str(declaration.get("file"))).resolve()
        try:
            source_path.relative_to(source_run_dir)
        except ValueError as exc:
            raise ValueError(f"Source artifact path escapes child run for {role}") from exc
        if len(matches) != 1 or matches[0].get("sha256") != declaration.get("sha256") or not source_path.is_file() or sha256_file(source_path) != declaration.get("sha256"):
            raise ValueError(f"Canonical source artifact lineage mismatch for {role}")
    return {
        "run_id": source_manifest["run_id"],
        "manifest_path": repo_relative_path(repo_root, manifest_path, "source child-run manifest"),
        "manifest_sha256": manifest_sha,
        "schema": source_manifest["schema"],
        "source_commit": source_manifest.get("source_commit"),
        "parent_manifests": parents,
    }


def validate_table_semantics(frames: dict[str, pd.DataFrame]) -> None:
    tear = frames["tear_sheet_factor_guard_ablation_ext2026.parquet"]
    equity = frames["factor_guard_ablation_equity_ext2026.parquet"]
    panel = frames["factor_guard_ablation_panel_ext2026.parquet"]
    if len(tear) != 21 or list(tear.get("row_order", ())) != list(range(21)):
        raise ValueError("Guard-ablation tear sheet must preserve the exact 21-row order")
    if set(tear.iloc[:12]["configuration"].astype(str)) != set(CONFIG_IDS):
        raise ValueError("Tear sheet must cover the exact four protocol cells")
    if tuple(equity.columns) != ("date", "configuration", "normalized_wealth", "drawdown", "relative_wealth", "relative_wealth_kind"):
        raise ValueError("Canonical equity table columns diverge")
    if set(panel["configuration"].astype(str)) != set(CONFIG_IDS):
        raise ValueError("Canonical panel must cover the exact four protocol cells")


def load_completed_bundle(repo_root: Path, bundle_dir: Path, source_run_dir: Path) -> tuple[dict, str, dict[str, pd.DataFrame], dict[str, dict], dict]:
    manifest_path, marker_path = bundle_dir / "manifest.json", bundle_dir / "COMPLETED"
    if not manifest_path.is_file() or not marker_path.is_file():
        raise FileNotFoundError(f"Completed guard-ablation bundle not found at {bundle_dir}")
    manifest_sha = sha256_file(manifest_path)
    if completed_manifest_sha256(marker_path) != manifest_sha:
        raise ValueError("Bundle COMPLETED marker does not exactly match manifest.json SHA-256")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("schema") != EXPECTED_BUNDLE_SCHEMA or manifest.get("completed") is not True or manifest.get("producer") != EXPECTED_PRODUCER:
        raise ValueError(f"Expected completed {EXPECTED_BUNDLE_SCHEMA} from {EXPECTED_PRODUCER}")
    source_child_run = validate_source_child_run_lineage(repo_root, source_run_dir, manifest)
    frames: dict[str, pd.DataFrame] = {}
    source_inventory: dict[str, dict] = {}
    for filename in EXPECTED_TABLE_SCHEMAS:
        _, meta = table_entry(manifest, filename)
        relative = Path(str(meta["file"]))
        path = (bundle_dir / relative).resolve()
        try:
            path.relative_to(bundle_dir)
        except ValueError as exc:
            raise ValueError(f"Manifest table path escapes bundle: {relative}") from exc
        frame = pd.read_parquet(path)
        source_inventory[filename] = validate_frame_inventory(repo_root, frame, path, meta, filename)
        frames[filename] = frame
    validate_table_semantics(frames)
    return manifest, manifest_sha, frames, source_inventory, source_child_run


def source_hash_snapshot(repo_root: Path, bundle_dir: Path, source_run_dir: Path, source_inventory: dict[str, dict], bundle_manifest: dict) -> dict[str, str]:
    snapshot = {
        "bundle/manifest.json": sha256_file(bundle_dir / "manifest.json"),
        "bundle/COMPLETED": sha256_file(bundle_dir / "COMPLETED"),
        "source_run/manifest.json": sha256_file(source_run_dir / "manifest.json"),
        "source_run/COMPLETED": sha256_file(source_run_dir / "COMPLETED"),
    }
    snapshot.update({f"table/{name}": sha256_file(repo_root / meta["file"]) for name, meta in source_inventory.items()})
    for role, declaration in bundle_manifest["source_artifacts"].items():
        snapshot[f"source_artifact/{role}"] = sha256_file(source_run_dir / declaration["file"])
    return snapshot


def assert_sources_unchanged(repo_root: Path, bundle_dir: Path, source_run_dir: Path, source_inventory: dict[str, dict], bundle_manifest: dict, before: dict[str, str]) -> None:
    if source_hash_snapshot(repo_root, bundle_dir, source_run_dir, source_inventory, bundle_manifest) != before:
        raise RuntimeError("Canonical guard-ablation inputs changed during presentation rendering")


def png_dimensions(path: Path) -> tuple[int, int]:
    with path.open("rb") as handle:
        header = handle.read(24)
    if len(header) != 24 or header[:8] != b"\x89PNG\r\n\x1a\n":
        raise ValueError(f"Not a PNG: {path}")
    return struct.unpack(">II", header[16:24])


def build_output_inventory(output_dir: Path, figure_names: tuple[str, ...]) -> dict[str, dict]:
    inventory: dict[str, dict] = {}
    for name in figure_names:
        path = output_dir / name
        width, height = png_dimensions(path)
        inventory[name] = {"file": name, "sha256": sha256_file(path), "bytes": path.stat().st_size, "width_px": width, "height_px": height, "media_type": "image/png"}
    return inventory


def write_presentation_manifest(*, repo_root: Path, notebook_path: Path, output_dir: Path, schema: str, appendix_id: str, bundle_dir: Path, bundle_manifest: dict, bundle_manifest_sha: str, source_inventory: dict[str, dict], source_child_run: dict, figure_names: tuple[str, ...]) -> Path:
    payload = {
        "schema": schema,
        "appendix_id": appendix_id,
        "completed": True,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "notebook": {"file": repo_relative_path(repo_root, notebook_path, "notebook source"), "sha256": sha256_file(notebook_path)},
        "source_bundle": {"path": repo_relative_path(repo_root, bundle_dir, "source bundle"), "schema": bundle_manifest["schema"], "report_id": bundle_manifest.get("report_id"), "manifest_sha256": bundle_manifest_sha, "completed_marker_sha256": sha256_file(bundle_dir / "COMPLETED")},
        "source_child_run": source_child_run,
        "source_artifacts": bundle_manifest["source_artifacts"],
        "source_tables": source_inventory,
        "outputs": build_output_inventory(output_dir, figure_names),
    }
    path = output_dir / "presentation_manifest.json"
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")
    return path


REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOK_PATH = resolve_repo_path(REPO_ROOT, os.environ.get("APPENDIX_F_NOTEBOOK_PATH", "notebooks/appendix_f_macro_to_bl_tilt.ipynb"), "APPENDIX_F_NOTEBOOK_PATH")
BUNDLE_DIR = resolve_repo_path(REPO_ROOT, os.environ.get("GUARD_ABLATION_BUNDLE_DIR", "data/provisional_remediation/canonical_guard_ablation_reports_v2"), "GUARD_ABLATION_BUNDLE_DIR")
SOURCE_RUN_DIR = resolve_repo_path(REPO_ROOT, os.environ.get("GUARD_ABLATION_SOURCE_RUN_DIR", "data/provisional_remediation/factor_guard_ablation_runs/factor_guard_ablation_ext2026_2019-01-01_2026-06-30_v2"), "GUARD_ABLATION_SOURCE_RUN_DIR")
OUTPUT_DIR = resolve_repo_path(REPO_ROOT, os.environ.get("APPENDIX_F_OUTPUT_DIR", "reports/appendix_f_macro_to_bl_tilt"), "APPENDIX_F_OUTPUT_DIR")
if any(OUTPUT_DIR == source or OUTPUT_DIR.is_relative_to(source) or source.is_relative_to(OUTPUT_DIR) for source in (BUNDLE_DIR, SOURCE_RUN_DIR)):
    raise ValueError("Appendix F output namespace must be separate from canonical input bundle and source run")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.family": "sans-serif", "font.size": 9, "axes.titlesize": 12, "axes.labelsize": 9, "axes.edgecolor": "#c3c2b7", "axes.linewidth": 0.8, "axes.facecolor": SURFACE, "figure.facecolor": SURFACE, "grid.color": GRID, "grid.linewidth": 0.6, "grid.linestyle": "-", "legend.frameon": False, "lines.linewidth": 2.0, "lines.solid_capstyle": "round"})
DIVERGING_CMAP = LinearSegmentedColormap.from_list("appendix_diverging", ["#2166ac", "#f0efec", "#b2182b"])
MACRO_Z_LIMIT = 3.0
PRESPECIFIED_MACRO_ASSET_PAIRS = (
    ("cpi_yoy_z", "IAU", "Inflation state ↔ gold allocation"),
    ("t10y2y_z", "SWDA.L", "Yield-curve state ↔ broad-equity allocation"),
    ("hy_oas_z", "BIL", "Credit-spread stress ↔ cash allocation"),
)
ASSET_COLORS = {"IAU": "#2a78d6", "SWDA.L": "#eb6834", "BIL": "#1baf7a"}
ASSET_MARKERS = {"IAU": "o", "SWDA.L": "s", "BIL": "^"}

In [ ]:
bundle_manifest, bundle_manifest_sha, frames, source_inventory, source_child_run = load_completed_bundle(
    REPO_ROOT, BUNDLE_DIR, SOURCE_RUN_DIR
)
source_hashes_before = source_hash_snapshot(
    REPO_ROOT, BUNDLE_DIR, SOURCE_RUN_DIR, source_inventory, bundle_manifest
)
summary = frames["tear_sheet_factor_guard_ablation_ext2026.parquet"].copy()
equity = frames["factor_guard_ablation_equity_ext2026.parquet"].copy()
panel_wide = frames["factor_guard_ablation_panel_ext2026.parquet"].copy()

panel_config_col = require_column(panel_wide, ("configuration",), "macro-to-BL panel")
date_col = require_column(panel_wide, ("rebalance_date", "date"), "macro-to-BL panel")
panel_wide[date_col] = pd.to_datetime(panel_wide[date_col])
if panel_wide.duplicated([panel_config_col, date_col]).any():
    raise ValueError("Canonical wide panel requires one row per configuration/rebalance date")

# Presentation-only reshape of producer-persisted asset columns. No allocation or
# financial value is calculated here; every long-form cell is copied verbatim.
ASSET_COLUMN_KEYS = {"SWDA.L": "SWDA_L", "XLK": "XLK", "IAU": "IAU", "BIL": "BIL"}
ASSET_VALUE_FIELDS = (
    "raw_tilt", "applied_tilt", "bl_q", "hrp_base_weight",
    "bl_weight", "target_weight", "target_delta",
)
required_asset_columns = {
    f"{field}_{key}" for field in ASSET_VALUE_FIELDS for key in ASSET_COLUMN_KEYS.values()
}
missing_asset_columns = sorted(required_asset_columns - set(panel_wide.columns))
if missing_asset_columns:
    raise ValueError(f"Canonical panel is missing producer-persisted asset mechanism fields: {missing_asset_columns}")
shared_columns = [column for column in panel_wide.columns if column not in required_asset_columns]
asset_rows = []
for source_row in panel_wide.to_dict("records"):
    shared = {column: source_row[column] for column in shared_columns}
    for asset_id, key in ASSET_COLUMN_KEYS.items():
        asset_rows.append({
            **shared,
            "asset_id": asset_id,
            **{field: source_row[f"{field}_{key}"] for field in ASSET_VALUE_FIELDS},
        })
panel = pd.DataFrame(asset_rows)
asset_col = "asset_id"
guarded = panel.loc[panel[panel_config_col].astype(str) == "factor_pit_ext2026"].copy()
if len(guarded) != 90 * len(ASSET_COLUMN_KEYS):
    raise ValueError("Primary guarded PIT mechanism projection must contain 90 dates × four assets")

display(Markdown(f"**Validated guarded-PIT source:** `{bundle_manifest.get('report_id', BUNDLE_DIR.name)}`  \\nManifest SHA-256: `{bundle_manifest_sha}`  \\nSource child run: `{source_child_run['run_id']}`"))
display(pd.DataFrame(source_inventory).T.reset_index(names="table")[["table", "schema", "rows", "sha256"]])

In [ ]:
macro_columns = ("cpi_yoy_z", "t10y2y_z", "hy_oas_z")
missing_macro = [name for name in macro_columns if name not in guarded.columns]
if missing_macro:
    raise ValueError(f"Canonical panel missing required macro axes: {missing_macro}")
macro_norm_col = require_column(guarded, ("macro_state_norm", "macro_z_norm"), "macro-to-BL panel")
for name in (*macro_columns, macro_norm_col):
    per_date_unique = guarded.groupby(date_col)[name].nunique(dropna=False)
    if (per_date_unique > 1).any():
        raise ValueError(f"Canonical macro field {name} conflicts across assets on a date")
macro_by_date = guarded[[date_col, *macro_columns, macro_norm_col]].drop_duplicates(date_col).sort_values(date_col)
macro_matrix = macro_by_date.set_index(date_col)[list(macro_columns)].T

fig, ax = plt.subplots(figsize=(14.5, 4.7), constrained_layout=True)
image = ax.imshow(macro_matrix.to_numpy(dtype=float), aspect="auto", interpolation="nearest",
                  cmap=DIVERGING_CMAP, vmin=-MACRO_Z_LIMIT, vmax=MACRO_Z_LIMIT)
ax.set_yticks(range(len(macro_columns)), labels=macro_columns)
step = max(1, len(macro_matrix.columns) // 12)
xticks = np.arange(0, len(macro_matrix.columns), step)
ax.set_xticks(xticks, labels=[macro_matrix.columns[i].strftime("%Y-%m") for i in xticks], rotation=45, ha="right")
ax.set_title("Guarded PIT macro state | fixed ±3 z-score scale, centered at zero", loc="left", fontweight="semibold")
ax.set_xlabel("Rebalance date")
colorbar = fig.colorbar(image, ax=ax, fraction=0.025, pad=0.02)
colorbar.set_label("Macro z-score (display clipped outside ±3; source table retained)")
ax.contour(macro_matrix.to_numpy(dtype=float), levels=[0.0], colors=INK, linewidths=0.35, alpha=0.75)
fig.savefig(OUTPUT_DIR / "appendix_f_macro_state_heatmap.png", dpi=180, bbox_inches="tight", facecolor=SURFACE)
plt.show()
display(Markdown("**Table view (CVD/print-safe signed values):**"))
display(macro_by_date.set_index(date_col)[list(macro_columns)])


In [ ]:
target_delta_col = require_column(guarded, ("target_delta", "persisted_target_delta"), "macro-to-BL panel")
if guarded.duplicated([date_col, asset_col]).any():
    raise ValueError("Canonical panel requires one guarded-PIT row per date/asset")
target_matrix = guarded.pivot(index=asset_col, columns=date_col, values=target_delta_col).sort_index()
if target_matrix.empty or not np.isfinite(target_matrix.to_numpy(dtype=float)).all():
    raise ValueError("Persisted target_delta heatmap requires a complete finite asset/date matrix")
target_delta_limit = float(np.nanmax(np.abs(target_matrix.to_numpy(dtype=float))))
if not np.isfinite(target_delta_limit) or target_delta_limit <= 0:
    raise ValueError("target_delta heatmap requires a positive symmetric plotting limit")

fig, ax = plt.subplots(figsize=(14.5, 5.2), constrained_layout=True)
image = ax.imshow(target_matrix.to_numpy(dtype=float), aspect="auto", interpolation="nearest",
                  cmap=DIVERGING_CMAP, vmin=-target_delta_limit, vmax=target_delta_limit)
ax.set_yticks(range(len(target_matrix.index)), labels=target_matrix.index)
step = max(1, len(target_matrix.columns) // 12)
xticks = np.arange(0, len(target_matrix.columns), step)
ax.set_xticks(xticks, labels=[target_matrix.columns[i].strftime("%Y-%m") for i in xticks], rotation=45, ha="right")
ax.set_title("Guarded PIT persisted final target_delta | symmetric zero-centered scale", loc="left", fontweight="semibold")
ax.set_xlabel("Rebalance date")
colorbar = fig.colorbar(image, ax=ax, fraction=0.025, pad=0.02)
colorbar.set_label("Final target-weight delta")
ax.contour(target_matrix.to_numpy(dtype=float), levels=[0.0], colors=INK, linewidths=0.35, alpha=0.75)
fig.savefig(OUTPUT_DIR / "appendix_f_target_delta_heatmap.png", dpi=180, bbox_inches="tight", facecolor=SURFACE)
plt.show()
display(Markdown("**Table view (CVD/print-safe signed values):**"))
display(target_matrix.T)


In [ ]:
association_candidates = (
    "model_implied_contemporaneous_association",
    "canonical_contemporaneous_association",
    "association_value",
)
association_col = next((name for name in association_candidates if name in guarded.columns), None)

fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.8), constrained_layout=True)
for ax, (macro_axis, asset_id, title) in zip(axes, PRESPECIFIED_MACRO_ASSET_PAIRS, strict=True):
    rows = guarded.loc[guarded[asset_col].astype(str) == asset_id].sort_values(macro_axis)
    if rows.empty:
        raise ValueError(f"Prespecified asset {asset_id} is absent for {macro_axis}")
    color = ASSET_COLORS[asset_id]
    marker = ASSET_MARKERS[asset_id]
    ax.scatter(rows[macro_axis], rows[target_delta_col], color=color, marker=marker, s=36,
               edgecolor=SURFACE, linewidth=1.3, alpha=0.82,
               label=f"{macro_axis} → {asset_id} target_delta")
    ax.axhline(0.0, color=SECONDARY, linewidth=0.8)
    ax.axvline(0.0, color=SECONDARY, linewidth=0.8)
    ax.grid(True)
    ax.set_title(title, loc="left", fontweight="semibold")
    ax.set_xlabel(macro_axis)
    ax.set_ylabel(f"{asset_id} persisted target_delta")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), fontsize=7.0)
    if association_col is not None:
        supplied = rows[association_col].dropna().unique()
        if len(supplied) > 1:
            raise ValueError(f"Producer supplied conflicting association values for {macro_axis}/{asset_id}")
        note = f"Model-implied contemporaneous association: {supplied[0]:.3f}\nNon-causal; producer-supplied." if len(supplied) == 1 else "No association value supplied."
    else:
        note = "No association value supplied; scatter shown descriptively."
    ax.text(0.02, 0.98, note, transform=ax.transAxes, fontsize=7.2, color=SECONDARY,
            va="top", bbox={"facecolor": SURFACE, "edgecolor": GRID, "alpha": 0.92, "pad": 3})
fig.suptitle("Prespecified macro-to-allocation links | descriptive, model-implied, contemporaneous, non-causal", fontsize=13, fontweight="semibold")
fig.savefig(OUTPUT_DIR / "appendix_f_macro_tilt_links.png", dpi=180, bbox_inches="tight", facecolor=SURFACE)
plt.show()

In [ ]:
loading_columns = (
    "loading_inflation",
    "loading_growth",
    "loading_credit_stress",
    "loading_policy",
    "loading_risk_appetite",
)
missing_loadings = [name for name in loading_columns if name not in guarded.columns]
if missing_loadings:
    raise ValueError(f"Canonical panel missing parsed loading fields: {missing_loadings}")
raw_tilt_col = require_column(guarded, ("raw_tilt", "raw_expected_excess_annualized"), "worked-date panel")
applied_tilt_col = require_column(guarded, ("applied_tilt", "guarded_tilt", "applied_expected_excess_annualized"), "worked-date panel")
bl_q_col = require_column(guarded, ("bl_q", "Q", "bl_view_q"), "worked-date panel")

# Prespecified worked-date rule: producer-owned macro-state norm only, never returns.
norm_by_date = macro_by_date.set_index(date_col)[macro_norm_col]
max_state_date = pd.Timestamp(norm_by_date.idxmax())
nearest_neutral_date = pd.Timestamp(norm_by_date.idxmin())
if max_state_date == nearest_neutral_date:
    raise ValueError("Worked-date selectors must produce two distinct dates")
selection_roles = {max_state_date: "maximum macro-state norm", nearest_neutral_date: "nearest-neutral macro-state norm"}
worked = guarded.loc[guarded[date_col].isin(selection_roles)].copy()
worked["worked_date_role"] = worked[date_col].map(selection_roles)
worked_columns = [
    "worked_date_role", date_col, asset_col, *macro_columns, macro_norm_col,
    *loading_columns, raw_tilt_col, applied_tilt_col, bl_q_col, target_delta_col,
]
worked_table = worked[worked_columns].sort_values([date_col, asset_col]).reset_index(drop=True)
display(Markdown("## Worked dates selected only by prespecified macro-state norm (not returns)"))
display(worked_table)

fig, axes = plt.subplots(2, 1, figsize=(17, 9.5), constrained_layout=True)
for ax, selected_date in zip(axes, (max_state_date, nearest_neutral_date), strict=True):
    shown = worked_table.loc[worked_table[date_col] == selected_date].drop(columns=["worked_date_role", date_col]).copy()
    shown = shown.apply(lambda col: col.map(lambda value: "" if pd.isna(value) else f"{value:.4f}" if isinstance(value, (float, np.floating)) else str(value)))
    ax.axis("off")
    ax.set_title(f"{selection_roles[selected_date]} — {selected_date.date()} | macro z → parsed loadings → raw tilt → applied tilt → BL Q → target_delta",
                 loc="left", fontsize=11, fontweight="semibold", pad=10)
    table = ax.table(cellText=shown.values, colLabels=shown.columns, loc="center", cellLoc="left", colLoc="left")
    table.auto_set_font_size(False)
    table.set_fontsize(6.1)
    table.scale(1.0, 1.45)
    for (row, _), cell in table.get_celld().items():
        cell.set_edgecolor(GRID)
        cell.set_linewidth(0.45)
        cell.set_facecolor("#f0efec" if row == 0 else SURFACE)
        cell.get_text().set_color(INK if row == 0 else SECONDARY)
fig.savefig(OUTPUT_DIR / "appendix_f_worked_dates.png", dpi=180, bbox_inches="tight", facecolor=SURFACE)
plt.show()


## Why parsed loadings and final BL tilts differ

The parsed macro loadings are not portfolio weights. The producer passes them
through a documented transformation chain:

1. the **exposure map** translates macro-axis loadings into asset-specific raw
   tilt directions and magnitudes;
2. model **conviction** scales those raw views;
3. guarded PIT applies **recall attenuation** using the persisted memorization
   evidence;
4. **BL conversion** maps the attenuated view into Black–Litterman `Q` and the
   posterior allocation response; and
5. the **HRP/BL blend** combines that response with the base portfolio, producing
   the persisted final `target_delta`.

Therefore a loading and a final tilt need not have equal magnitude, and can be
muted by multiple declared stages. The small multiples are descriptive,
model-implied contemporaneous associations and are explicitly **non-causal**.
Association values are displayed only when supplied by the canonical producer;
this notebook does not calculate correlations or regressions.


In [ ]:
F_FIGURES = (
    "appendix_f_macro_state_heatmap.png",
    "appendix_f_target_delta_heatmap.png",
    "appendix_f_macro_tilt_links.png",
    "appendix_f_worked_dates.png",
)
assert_sources_unchanged(
    REPO_ROOT, BUNDLE_DIR, SOURCE_RUN_DIR, source_inventory, bundle_manifest, source_hashes_before
)
presentation_manifest_path = write_presentation_manifest(
    repo_root=REPO_ROOT,
    notebook_path=NOTEBOOK_PATH,
    output_dir=OUTPUT_DIR,
    schema="appendix_f_macro_to_bl_tilt.presentation.v1",
    appendix_id="appendix_f_macro_to_bl_tilt",
    bundle_dir=BUNDLE_DIR,
    bundle_manifest=bundle_manifest,
    bundle_manifest_sha=bundle_manifest_sha,
    source_inventory=source_inventory,
    source_child_run=source_child_run,
    figure_names=F_FIGURES,
)
assert_sources_unchanged(
    REPO_ROOT, BUNDLE_DIR, SOURCE_RUN_DIR, source_inventory, bundle_manifest, source_hashes_before
)
display(Markdown(f"**Appendix F presentation manifest:** `{presentation_manifest_path}`  \\nSHA-256: `{sha256_file(presentation_manifest_path)}`"))